In [ ]:
import json
import logging
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime
from contextlib import contextmanager

import cv2
import pytesseract
from groq import Groq
from dotenv import load_dotenv
from pdf2image import convert_from_path
import PyPDF2
import fitz  # PyMuPDF


@dataclass
class TextRegion:
    """Represents a text region with its metadata."""
    text: str
    x: int
    y: int
    width: int
    height: int
    font_size: Optional[float] = None
    font_name: Optional[str] = None
    confidence: Optional[float] = None


@dataclass
class TableBounds:
    """Represents the bounding box of a detected table."""
    x: int
    y: int
    width: int
    height: int


@dataclass
class ExtractedTable:
    """Represents an extracted table with its content."""
    bounds: TableBounds
    raw_text: str
    structured_data: Dict[str, Any]
    page_number: int
    table_type: str


@dataclass
class PageContent:
    """Represents content from a single page."""
    page_number: int
    raw_text: str
    cleaned_text: str
    structured_text: List[TextRegion]
    tables: List[ExtractedTable]
    word_count: int
    character_count: int
    language: str
    extraction_method: str
    medical_sections: Dict[str, str]


@dataclass
class MedicalMetadata:
    """Medical-specific metadata - generalized approach."""
    patient_info: Dict[str, str]
    report_type: str
    report_date: Optional[str]
    doctor_info: Dict[str, str]
    lab_info: Dict[str, str]
    test_parameters: List[str]
    extracted_fields: Dict[str, str]  # Generic extracted fields


@dataclass
class DocumentMetadata:
    """Represents document-level metadata."""
    filename: str
    file_size: int
    total_pages: int
    creation_date: Optional[str]
    modification_date: Optional[str]
    extraction_timestamp: str
    total_words: int
    total_characters: int
    total_tables: int
    medical_metadata: MedicalMetadata


class GeneralizedMedicalPDFExtractor:
    """
    A generalized medical PDF extraction tool that extracts both text and tables
    from any medical reports and formats them into structured JSON without
    making assumptions about medical content.
    """

    def __init__(self, pdf_path: str, groq_model: str = "deepseek-r1-distill-llama-70b"):
        """
        Initialize the generalized medical PDF extractor.
        
        Args:
            pdf_path: Path to the PDF file to process
            groq_model: Groq model to use for text processing
        """
        self.pdf_path = Path(pdf_path)
        self.groq_model = groq_model
        self._setup_logging()
        self._setup_groq_client()
        self._temp_images: List[str] = []
        self.document_metadata: Optional[DocumentMetadata] = None
        self.pages_content: List[PageContent] = []

        # Generalized patterns for any medical document
        self.extraction_patterns = {
            "patient_identifier": [
                r"(?:Patient\s*(?:Name|ID)|Name|ID|MRN|Medical\s*Record)\s*:?\s*([A-Za-z0-9\s\.\-]+)",
                r"(?:Mr\.|Mrs\.|Ms\.|Dr\.)\s*([A-Za-z\s\.]+)"
            ],
            "dates": [
                r"(?:Date|Report\s*Date|Test\s*Date|Collection\s*Date)\s*:?\s*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})",
                r"(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})"
            ],
            "doctor_info": [
                r"(?:Doctor|Physician|Dr\.)\s*:?\s*([A-Za-z\s\.]+)",
                r"(?:Referred\s*by|Reporting\s*Doctor)\s*:?\s*([A-Za-z\s\.]+)"
            ],
            "numeric_values": [
                r"(\w+(?:\s+\w+)*)\s*:?\s*(\d+(?:\.\d+)?)\s*([A-Za-z\/\%]*)",
                r"(\w+)\s+(\d+(?:\.\d+)?)\s+([A-Za-z\/\%]+)"
            ],
            "test_parameters": [
                r"^([A-Z][A-Za-z\s\(\)]+)\s+(\d+(?:\.\d+)?)",
                r"(\w+(?:\s+\w+)*)\s*:\s*(\d+(?:\.\d+)?)"
            ]
        }

    def _setup_logging(self) -> None:
        """Configure logging for the extractor."""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('generalized_medical_extractor.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def _setup_groq_client(self) -> None:
        """Initialize the Groq client with API key from environment."""
        load_dotenv()
        groq_api_key = os.getenv("GROQ_API_KEY")

        if not groq_api_key:
            self.logger.warning("GROQ_API_KEY not found. AI-powered processing will be disabled.")
            self.groq_client = None
        else:
            self.groq_client = Groq(api_key=groq_api_key)

    @contextmanager
    def _cleanup_temp_files(self):
        """Context manager to ensure temporary files are cleaned up."""
        try:
            yield
        finally:
            self._remove_temp_images()

    def _remove_temp_images(self) -> None:
        """Remove temporary image files."""
        for image_path in self._temp_images:
            try:
                Path(image_path).unlink(missing_ok=True)
            except Exception as e:
                self.logger.warning(f"Failed to remove temp file {image_path}: {e}")
        self._temp_images.clear()

    def _fix_generalized_ocr_corruption(self, text: str) -> str:
        """
        Generalized OCR corruption fixes that work across all medical reports.
        Only fixes obvious OCR artifacts, doesn't make medical assumptions.
        """
        if not text:
            return text
        
        # Common OCR character substitutions (language-agnostic)
        ocr_char_fixes = [
            # Common OCR character confusion
            (r'(?<=\d)O(?=\d)', '0'),  # O -> 0 when between digits
            (r'(?<=\d)l(?=\d)', '1'),  # l -> 1 when between digits
            (r'(?<=\d)I(?=\d)', '1'),  # I -> 1 when between digits
            (r'(?<=\d)S(?=\d)', '5'),  # S -> 5 when between digits
            
            # Fix common spacing issues around numbers and symbols
            (r'(\d)\s*-\s*(\d)', r'\1-\2'),  # Fix spaces around dashes in ranges
            (r'(\d)\s*\.\s*(\d)', r'\1.\2'),  # Fix spaces around decimal points
            (r'(\d)\s*/\s*(\w)', r'\1/\2'),   # Fix spaces around forward slashes
            
            # Remove extra spaces
            (r'\s+', ' '),  # Multiple spaces -> single space
            (r'^\s+|\s+$', ''),  # Trim leading/trailing spaces
        ]
        
        result = text
        for pattern, replacement in ocr_char_fixes:
            result = re.sub(pattern, replacement, result)
        
        return result

    def _extract_structured_data_from_text(self, text: str) -> List[Dict[str, Any]]:
        """
        Extract structured data from OCR text without making medical assumptions.
        Works for any tabular medical data.
        """
        structured_data = []
        lines = text.strip().split('\n')
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            # Pattern 1: Parameter | Value | Unit | Reference (pipe-separated)
            pipe_pattern = r'^([^|]+)\|([^|]+)\|([^|]*)\|([^|]*)$'
            pipe_match = re.match(pipe_pattern, line)
            if pipe_match:
                structured_data.append({
                    'parameter': pipe_match.group(1).strip(),
                    'value': pipe_match.group(2).strip(),
                    'unit': pipe_match.group(3).strip() if pipe_match.group(3).strip() else None,
                    'reference_range': pipe_match.group(4).strip() if pipe_match.group(4).strip() else None
                })
                continue
            
            # Pattern 2: Parameter followed by numeric value and optional unit/range
            # Example: "SERUM BILIRUBIN (TOTAL) 0.9 mg/dl 0.2 - 1.2"
            param_value_pattern = r'^([A-Za-z\s\(\)]+?)\s+(\d+(?:\.\d+)?)\s*([A-Za-z/]+)?\s*((?:\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?.*?)?)$'
            param_match = re.match(param_value_pattern, line)
            if param_match:
                parameter = param_match.group(1).strip()
                value = param_match.group(2).strip()
                unit = param_match.group(3).strip() if param_match.group(3) else None
                reference = param_match.group(4).strip() if param_match.group(4) else None
                
                structured_data.append({
                    'parameter': parameter,
                    'value': value,
                    'unit': unit,
                    'reference_range': reference
                })
                continue
            
            # Pattern 3: Colon-separated format
            # Example: "Parameter: Value Unit Reference"
            colon_pattern = r'^([^:]+):\s*(\d+(?:\.\d+)?)\s*([A-Za-z/]+)?\s*(.*?)$'
            colon_match = re.match(colon_pattern, line)
            if colon_match:
                structured_data.append({
                    'parameter': colon_match.group(1).strip(),
                    'value': colon_match.group(2).strip(),
                    'unit': colon_match.group(3).strip() if colon_match.group(3) else None,
                    'reference_range': colon_match.group(4).strip() if colon_match.group(4) else None
                })
                continue
            
            # Pattern 4: Tab-separated or multiple space-separated
            parts = re.split(r'\t+|\s{2,}', line)
            if len(parts) >= 2:
                # First part is parameter, try to identify value, unit, reference
                parameter = parts[0].strip()
                
                # Look for numeric value in the parts
                value = None
                unit = None
                reference_range = None
                
                for i, part in enumerate(parts[1:], 1):
                    part = part.strip()
                    if not part:
                        continue
                        
                    # Check if this part contains a number (likely the value)
                    if re.search(r'\d+(?:\.\d+)?', part) and value is None:
                        # Extract just the numeric part
                        num_match = re.search(r'(\d+(?:\.\d+)?)', part)
                        if num_match:
                            value = num_match.group(1)
                            # Check if there's a unit attached
                            remaining = part.replace(value, '').strip()
                            if remaining and not re.search(r'\d', remaining):
                                unit = remaining
                    
                    # Check if this part looks like a unit
                    elif re.match(r'^[A-Za-z/]+$', part) and unit is None:
                        unit = part
                    
                    # Check if this part looks like a reference range
                    elif '-' in part or 'to' in part.lower():
                        reference_range = part
                
                if parameter and value:
                    structured_data.append({
                        'parameter': parameter,
                        'value': value,
                        'unit': unit,
                        'reference_range': reference_range
                    })
        
        return structured_data

    def _extract_pdf_metadata(self) -> DocumentMetadata:
        """Extract metadata from PDF file."""
        try:
            file_stats = self.pdf_path.stat()
            
            # Initialize medical metadata
            medical_metadata = MedicalMetadata(
                patient_info={},
                report_type="",
                report_date=None,
                doctor_info={},
                lab_info={},
                test_parameters=[],
                extracted_fields={}
            )
            
            # Try PyPDF2 first
            try:
                with open(self.pdf_path, 'rb') as file:
                    pdf_reader = PyPDF2.PdfReader(file)
                    
                    return DocumentMetadata(
                        filename=self.pdf_path.name,
                        file_size=file_stats.st_size,
                        total_pages=len(pdf_reader.pages),
                        creation_date='',
                        modification_date='',
                        extraction_timestamp=datetime.now().isoformat(),
                        total_words=0,
                        total_characters=0,
                        total_tables=0,
                        medical_metadata=medical_metadata
                    )
            except Exception as e:
                self.logger.warning(f"PyPDF2 metadata extraction failed: {e}")
                
            # Basic fallback
            return DocumentMetadata(
                filename=self.pdf_path.name,
                file_size=file_stats.st_size,
                total_pages=0,
                creation_date='',
                modification_date='',
                extraction_timestamp=datetime.now().isoformat(),
                total_words=0,
                total_characters=0,
                total_tables=0,
                medical_metadata=medical_metadata
            )
            
        except Exception as e:
            self.logger.error(f"Error extracting PDF metadata: {e}")
            raise

    def _preprocess_image(self, image_path: str) -> cv2.Mat:
        """Preprocess image for better table detection."""
        try:
            img = cv2.imread(image_path)
            if img is None:
                raise FileNotFoundError(f"Could not read image: {image_path}")
            
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            denoised = cv2.fastNlMeansDenoising(gray, None, 30, 7, 21)
            binary = cv2.adaptiveThreshold(
                denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                cv2.THRESH_BINARY, 11, 2
            )
            
            return binary
            
        except Exception as e:
            self.logger.error(f"Error preprocessing image {image_path}: {e}")
            raise

    def _detect_tables(self, image: cv2.Mat) -> List[TableBounds]:
        """Detect tables in the preprocessed image."""
        try:
            height, width = image.shape
            
            vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, height // 30))
            horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (width // 30, 1))
            
            vertical_lines = cv2.morphologyEx(image, cv2.MORPH_OPEN, vertical_kernel, iterations=2)
            horizontal_lines = cv2.morphologyEx(image, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
            
            table_mask = cv2.add(horizontal_lines, vertical_lines)
            contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            tables = []
            min_area = width * height * 0.01  # Increased minimum area
            
            for contour in contours:
                x, y, w, h = cv2.boundingRect(contour)
                if w * h > min_area and w > 50 and h > 50:
                    tables.append(TableBounds(x, y, w, h))
            
            return tables
            
        except Exception as e:
            self.logger.error(f"Error detecting tables: {e}")
            return []

    def _extract_text_from_table(self, image: cv2.Mat, table: TableBounds) -> str:
        """Extract text from a specific table region."""
        try:
            table_image = image[
                table.y:table.y + table.height,
                table.x:table.x + table.width
            ]
            
            custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz .,()-+=%/:'
            text = pytesseract.image_to_string(table_image, config=custom_config, lang='eng')
            
            # Apply generalized OCR fixes
            fixed_text = self._fix_generalized_ocr_corruption(text.strip())
            
            return fixed_text
            
        except Exception as e:
            self.logger.error(f"Error extracting text from table: {e}")
            return ""

    def _classify_table_type_generalized(self, text: str) -> str:
        """
        Classify table type based on content without hardcoded medical categories.
        Returns generic descriptive types.
        """
        text_lower = text.lower()
        
        # Look for keywords to suggest table type
        if any(word in text_lower for word in ['test', 'result', 'value', 'parameter']):
            return "test_results_table"
        elif any(word in text_lower for word in ['medication', 'drug', 'dosage', 'dose']):
            return "medication_table"
        elif any(word in text_lower for word in ['vital', 'sign', 'pressure', 'temperature']):
            return "vital_signs_table"
        elif any(word in text_lower for word in ['history', 'complaint', 'symptom']):
            return "clinical_history_table"
        elif any(word in text_lower for word in ['recommendation', 'advice', 'instruction']):
            return "recommendations_table"
        else:
            return "general_medical_table"

    def _format_generalized_table_as_json(self, text: str, table_type: str = "medical_table") -> Dict[str, Any]:
        """
        Format any medical table as JSON without hardcoded medical knowledge.
        Uses only the information present in the OCR text.
        """
        if not self.groq_client or not text:
            # Fallback: extract structured data without AI
            extracted_data = self._extract_structured_data_from_text(text)
            return {
                "raw_text": text,
                "structured_data": {item['parameter']: {
                    'value': item['value'],
                    'unit': item['unit'],
                    'reference_range': item['reference_range']
                } for item in extracted_data if item.get('parameter') and item.get('value')},
                "parsing_status": "rule_based_extraction"
            }
        
        try:
            # Fix only obvious OCR corruption
            cleaned_text = self._fix_generalized_ocr_corruption(text)
            
            prompt = f"""
            You are a data extraction assistant. Extract structured information from this medical table text.
            
            RULES:
            1. Extract ONLY the information that is clearly present in the text
            2. Do NOT add any medical knowledge, reference ranges, or interpretations
            3. Do NOT correct values based on medical knowledge
            4. If the text says a value is X, then X is correct regardless of medical plausibility
            5. Preserve all numbers, units, and ranges exactly as written
            6. If reference ranges are missing or unclear, leave them as null
            7.NEVER remove decimal points from numbers
            8.For this medical report, extract:
            - Test parameter names
            - Test values (preserve ALL decimal points)
            - Units 
            - Reference ranges (preserve ALL decimal points)
            9. Do not remove trailing zeroes from decimal values
            10. Do not round numbers, preserve exact values as written
            11. Structure the data as: parameter -> {{value, unit, reference_range}}
            
            Table Type: {table_type}
            
            Text to extract from:
            {cleaned_text}
            
            Return JSON with this structure:
            {{
                "parameter_name": {{
                    "value": "exact_value_from_text",
                    "unit": "unit_if_present",
                    "reference_range": "range_if_present_or_null"
                }}
            }}
            
            Extract only what is explicitly written in the text.
            """
            
            response = self.groq_client.chat.completions.create(
                model=self.groq_model,
                messages=[
                    {"role": "system", "content": "You are a precise data extraction assistant. You extract only the information explicitly present in the text without making assumptions or corrections based on domain knowledge."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,  # Lower temperature for more consistent extraction
                max_tokens=2000
            )
            
            if hasattr(response, 'choices') and len(response.choices) > 0:
                content = response.choices[0].message.content.strip()
                extracted_json = self._extract_json_from_response(content)
                if extracted_json:
                    return {
                        "raw_text": text,
                        "structured_data": extracted_json,
                        "parsing_status": "ai_enhanced_extraction"
                    }
            
            # Fallback to rule-based extraction
            extracted_data = self._extract_structured_data_from_text(cleaned_text)
            return {
                "raw_text": text,
                "structured_data": {item['parameter']: {
                    'value': item['value'],
                    'unit': item['unit'],
                    'reference_range': item['reference_range']
                } for item in extracted_data if item.get('parameter') and item.get('value')},
                "parsing_status": "fallback_rule_based"
            }
            
        except Exception as e:
            self.logger.error(f"Error in generalized table formatting: {e}")
            return {"raw_text": text, "error": str(e), "parsing_status": "extraction_failed"}

    def _extract_json_from_response(self, response: str) -> Optional[Dict[str, Any]]:
        """Extract JSON object from API response."""
        if not response:
            return None
            
        try:
            # Try to find JSON in code blocks first
            json_pattern = r'```json\s*\n(.*?)\n\s*```'
            match = re.search(json_pattern, response, re.DOTALL)
            
            if match:
                json_str = match.group(1).strip()
            else:
                # Try to find JSON without code blocks
                json_pattern = r'\{.*\}'
                match = re.search(json_pattern, response, re.DOTALL)
                if match:
                    json_str = match.group(0)
                else:
                    json_str = response.strip()
            
            # Clean up the JSON string
            json_str = json_str.strip()
            if json_str.startswith('```json'):
                json_str = json_str[7:]
            if json_str.endswith('```'):
                json_str = json_str[:-3]
            json_str = json_str.strip()
            
            return json.loads(json_str)
            
        except json.JSONDecodeError as e:
            self.logger.warning(f"Failed to parse JSON from response: {e}")
            return None
        except Exception as e:
            self.logger.error(f"Unexpected error parsing JSON: {e}")
            return None

    def _extract_generalized_medical_info(self, text: str) -> MedicalMetadata:
        """Extract medical-specific information from text using generalized patterns."""
        medical_metadata = MedicalMetadata(
            patient_info={},
            report_type="",
            report_date=None,
            doctor_info={},
            lab_info={},
            test_parameters=[],
            extracted_fields={}
        )
        
        # Extract patient information using flexible patterns
        for pattern in self.extraction_patterns["patient_identifier"]:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                for i, match in enumerate(matches):
                    medical_metadata.patient_info[f"identifier_{i+1}"] = match.strip()
        
        # Extract dates
        for pattern in self.extraction_patterns["dates"]:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                medical_metadata.report_date = matches[0] if matches else None
                for i, match in enumerate(matches):
                    medical_metadata.extracted_fields[f"date_{i+1}"] = match.strip()
        
        # Extract doctor information
        for pattern in self.extraction_patterns["doctor_info"]:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                for i, match in enumerate(matches):
                    medical_metadata.doctor_info[f"doctor_{i+1}"] = match.strip()
        
        # Extract test parameters and values
        for pattern in self.extraction_patterns["numeric_values"]:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                if len(match) >= 2:
                    param_name = match[0].strip()
                    medical_metadata.test_parameters.append(param_name)
                    medical_metadata.extracted_fields[param_name] = {
                        "value": match[1].strip() if len(match) > 1 else "",
                        "unit": match[2].strip() if len(match) > 2 else ""
                    }
        
        # Determine report type based on content analysis (generalized)
        text_lower = text.lower()
        if "blood" in text_lower or "serum" in text_lower:
            medical_metadata.report_type = "blood_test_report"
        elif "urine" in text_lower:
            medical_metadata.report_type = "urine_test_report"
        elif "x-ray" in text_lower or "radiolog" in text_lower:
            medical_metadata.report_type = "radiology_report"
        elif "patholog" in text_lower or "biopsy" in text_lower:
            medical_metadata.report_type = "pathology_report"
        elif "echo" in text_lower or "cardiac" in text_lower:
            medical_metadata.report_type = "cardiac_report"
        else:
            medical_metadata.report_type = "general_medical_report"
        
        return medical_metadata

    def _clean_text(self, text: str) -> str:
        """Clean and normalize extracted text."""
        if not text:
            return ""
        
        cleaned = re.sub(r'[ \t]+', ' ', text)
        cleaned = re.sub(r'\n\s*\n', '\n', cleaned)
        cleaned = '\n'.join(line.strip() for line in cleaned.split('\n') if line.strip())
        
        return cleaned

    def _process_page_generalized(self, page_num: int) -> PageContent:
        """
        Process a page with generalized approach - no hardcoded medical knowledge.
        """
        try:
            # Convert page to image for table detection
            images = convert_from_path(str(self.pdf_path), first_page=page_num+1, last_page=page_num+1, dpi=300)
            if not images:
                raise ValueError(f"Could not convert page {page_num + 1} to image")
            
            image = images[0]
            image_path = f'temp_page_{page_num}_{self.pdf_path.stem}.jpg'
            image.save(image_path, 'JPEG', quality=95)
            self._temp_images.append(image_path)
            
            # Extract text using PyMuPDF
            with fitz.open(self.pdf_path) as doc:
                page = doc[page_num]
                
                raw_text = page.get_text()
                cleaned_text = self._clean_text(raw_text)
                
                # Extract structured text regions
                text_dict = page.get_text("dict")
                structured_text = []
                
                for block in text_dict["blocks"]:
                    if "lines" in block:
                        for line in block["lines"]:
                            for span in line["spans"]:
                                if span["text"].strip():
                                    structured_text.append(TextRegion(
                                        text=span["text"],
                                        x=int(span["bbox"][0]),
                                        y=int(span["bbox"][1]),
                                        width=int(span["bbox"][2] - span["bbox"][0]),
                                        height=int(span["bbox"][3] - span["bbox"][1]),
                                        font_size=span["size"],
                                        font_name=span["font"]
                                    ))
            
            # Detect and extract tables
            preprocessed = self._preprocess_image(image_path)
            table_bounds = self._detect_tables(preprocessed)
            
            tables = []
            for table_num, table in enumerate(table_bounds):
                raw_table_text = self._extract_text_from_table(preprocessed, table)
                if raw_table_text:
                    table_type = self._classify_table_type_generalized(raw_table_text)
                    structured_data = self._format_generalized_table_as_json(raw_table_text, table_type)
                    
                    extracted_table = ExtractedTable(
                        bounds=table,
                        raw_text=raw_table_text,
                        structured_data=structured_data,
                        page_number=page_num + 1,
                        table_type=table_type
                    )
                    tables.append(extracted_table)
                                # Extract medical sections using generalized patterns
            medical_sections = self._extract_medical_sections_generalized(cleaned_text)
            
            return PageContent(
                page_number=page_num + 1,
                raw_text=raw_text,
                cleaned_text=cleaned_text,
                structured_text=structured_text,
                tables=tables,
                word_count=len(cleaned_text.split()),
                character_count=len(cleaned_text),
                language="en",  # Default to English, could be enhanced with language detection
                extraction_method="pymupdf_with_table_detection",
                medical_sections=medical_sections
            )
            
        except Exception as e:
            self.logger.error(f"Error processing page {page_num + 1}: {e}")
            # Return empty page content on error
            return PageContent(
                page_number=page_num + 1,
                raw_text="",
                cleaned_text="",
                structured_text=[],
                tables=[],
                word_count=0,
                character_count=0,
                language="en",
                extraction_method="error_fallback",
                medical_sections={}
            )

    def _extract_medical_sections_generalized(self, text: str) -> Dict[str, str]:
        """
        Extract common medical sections from text using generalized patterns.
        """
        sections = {}
        
        # Common medical section headers (generalized)
        section_patterns = {
            "patient_demographics": [
                r"(?i)(patient\s+(?:information|details|demographics)[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\n\s*\w+\s+\w+\s*:|\Z)",
                r"(?i)(name\s*:[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ],
            "clinical_history": [
                r"(?i)((?:clinical\s+)?history[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(chief\s+complaint[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ],
            "examination": [
                r"(?i)((?:physical\s+)?examination[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(clinical\s+findings[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ],
            "test_results": [
                r"(?i)((?:test\s+)?results?[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(laboratory\s+(?:results?|findings)[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ],
            "diagnosis": [
                r"(?i)(diagnosis[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(impression[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ],
            "recommendations": [
                r"(?i)(recommendations?[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(advice[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)",
                r"(?i)(treatment\s+plan[\s\S]*?)(?=\n\s*[A-Z][^a-z]*:|\Z)"
            ]
        }
        
        for section_name, patterns in section_patterns.items():
            for pattern in patterns:
                match = re.search(pattern, text, re.MULTILINE | re.DOTALL)
                if match:
                    sections[section_name] = match.group(1).strip()
                    break  # Use first matching pattern
        
        return sections

    def extract_complete_document(self) -> Dict[str, Any]:
        """
        Extract complete document with all pages, tables, and metadata.
        
        Returns:
            Complete document data as a dictionary
        """
        self.logger.info(f"Starting extraction of {self.pdf_path}")
        
        with self._cleanup_temp_files():
            try:
                # Extract document metadata
                self.document_metadata = self._extract_pdf_metadata()
                
                # Process each page
                with fitz.open(self.pdf_path) as doc:
                    total_pages = len(doc)
                    self.document_metadata.total_pages = total_pages
                    
                    for page_num in range(total_pages):
                        self.logger.info(f"Processing page {page_num + 1}/{total_pages}")
                        page_content = self._process_page_generalized(page_num)
                        self.pages_content.append(page_content)
                
                # Update document metadata with aggregated data
                self._update_document_metadata()
                
                # Create complete document structure
                complete_document = {
                    "document_metadata": asdict(self.document_metadata),
                    "pages": [asdict(page) for page in self.pages_content],
                    "extraction_summary": self._generate_extraction_summary()
                }
                
                self.logger.info("Document extraction completed successfully")
                return complete_document
                
            except Exception as e:
                self.logger.error(f"Error during document extraction: {e}")
                raise

    def _update_document_metadata(self) -> None:
        """Update document metadata with aggregated information from all pages."""
        if not self.document_metadata:
            return
        
        total_words = sum(page.word_count for page in self.pages_content)
        total_characters = sum(page.character_count for page in self.pages_content)
        total_tables = sum(len(page.tables) for page in self.pages_content)
        
        self.document_metadata.total_words = total_words
        self.document_metadata.total_characters = total_characters
        self.document_metadata.total_tables = total_tables
        
        # Extract medical metadata from all text
        all_text = " ".join(page.cleaned_text for page in self.pages_content)
        self.document_metadata.medical_metadata = self._extract_generalized_medical_info(all_text)

    def _generate_extraction_summary(self) -> Dict[str, Any]:
        """Generate a summary of the extraction process."""
        return {
            "total_pages_processed": len(self.pages_content),
            "total_tables_found": sum(len(page.tables) for page in self.pages_content),
            "pages_with_tables": len([page for page in self.pages_content if page.tables]),
            "extraction_methods_used": list(set(page.extraction_method for page in self.pages_content)),
            "average_words_per_page": sum(page.word_count for page in self.pages_content) / len(self.pages_content) if self.pages_content else 0,
            "languages_detected": list(set(page.language for page in self.pages_content)),
            "medical_sections_found": list(set().union(*(page.medical_sections.keys() for page in self.pages_content))),
            "ai_processing_available": self.groq_client is not None
        }

    def save_extracted_data(self, output_path: str) -> None:
        """
        Save extracted data to JSON file.
        
        Args:
            output_path: Path where to save the JSON file
        """
        try:
            complete_data = self.extract_complete_document()
            
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(complete_data, f, indent=2, ensure_ascii=False)
            
            self.logger.info(f"Extracted data saved to {output_path}")
            
        except Exception as e:
            self.logger.error(f"Error saving extracted data: {e}")
            raise

    def get_tables_only(self) -> List[Dict[str, Any]]:
        """
        Extract only tables from the document.
        
        Returns:
            List of all tables found in the document
        """
        if not self.pages_content:
            self.extract_complete_document()
        
        all_tables = []
        for page in self.pages_content:
            for table in page.tables:
                all_tables.append(asdict(table))
        
        return all_tables

    def get_medical_summary(self) -> Dict[str, Any]:
        """
        Get a medical summary of the document.
        
        Returns:
            Dictionary containing medical summary information
        """
        if not self.document_metadata:
            self.extract_complete_document()
        
        return {
            "document_info": {
                "filename": self.document_metadata.filename,
                "total_pages": self.document_metadata.total_pages,
                "report_type": self.document_metadata.medical_metadata.report_type,
                "report_date": self.document_metadata.medical_metadata.report_date
            },
            "patient_info": self.document_metadata.medical_metadata.patient_info,
            "doctor_info": self.document_metadata.medical_metadata.doctor_info,
            "test_parameters": self.document_metadata.medical_metadata.test_parameters,
            "extracted_fields": self.document_metadata.medical_metadata.extracted_fields,
            "total_tables": self.document_metadata.total_tables,
            "medical_sections": list(set().union(*(page.medical_sections.keys() for page in self.pages_content)))
        }


# Example usage and main function
def main(pdf_path=None, output="extracted_data.json", tables_only=False, 
         summary_only=False, groq_model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Jupyter-friendly main function."""
    
    if pdf_path is None:
        print("Please provide a PDF path")
        return
    
    try:
        # Initialize extractor
        extractor = GeneralizedMedicalPDFExtractor(pdf_path, groq_model)
        
        if tables_only:
            # Extract only tables
            tables = extractor.get_tables_only()
            with open(output, 'w', encoding='utf-8') as f:
                json.dump({"tables": tables}, f, indent=2, ensure_ascii=False)
            print(f"Tables extracted and saved to {output}")
            
        elif summary_only:
            # Extract only medical summary
            summary = extractor.get_medical_summary()
            with open(output, 'w', encoding='utf-8') as f:
                json.dump(summary, f, indent=2, ensure_ascii=False)
            print(f"Medical summary extracted and saved to {output}")
            
        else:
            # Extract complete document
            extractor.save_extracted_data(output)
            print(f"Complete document extracted and saved to {output}")
            
    except Exception as e:
        print(f"Error: {e}")
        return 1
    
    return 0

# Remove the argparse version and use this instead:
if __name__ == "__main__":
    # For Jupyter usage, just define your parameters here:
    main(
        pdf_path="cbc-report-format.pdf",  # Change this path
        output="extracted_data.json",
        tables_only=False,
        summary_only=False
    )



2025-12-20 04:11:41,919 - INFO - Starting extraction of cbc-report-format.pdf
2025-12-20 04:11:41,931 - INFO - Processing page 1/2
2025-12-20 04:11:54,141 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:11:57,225 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:11:58,408 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:11:58,425 - INFO - Processing page 2/2
2025-12-20 04:12:03,272 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:12:05,182 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:12:07,905 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:12:07,914 - INFO - Document extraction completed successfully
2025-12-20 04:12:07,929 - INFO -

Complete document extracted and saved to extracted_data.json


In [ ]:
import json
import logging
import os
import re
from pathlib import Path
from typing import Dict, Any, Optional
from datetime import datetime

import fitz  # PyMuPDF
from groq import Groq
from dotenv import load_dotenv


class StreamlinedMedicalOCR:
    """
    Streamlined medical OCR that extracts only essential information.
    Removes redundant data and focuses on actionable medical information.
    """

    def __init__(self, pdf_path: str, groq_model: str = "meta-llama/llama-4-scout-17b-16e-instruct"):
        self.pdf_path = Path(pdf_path)
        self.groq_model = groq_model
        self._setup_logging()
        self._setup_groq_client()

    def _setup_logging(self) -> None:
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)

    def _setup_groq_client(self) -> None:
        load_dotenv()
        groq_api_key = os.getenv("GROQ_API_KEY")
        
        if not groq_api_key:
            self.logger.warning("GROQ_API_KEY not found. Using rule-based extraction only.")
            self.groq_client = None
        else:
            self.groq_client = Groq(api_key=groq_api_key)

    def _extract_full_text(self) -> str:
        """Extract all text from PDF."""
        full_text = ""
        with fitz.open(self.pdf_path) as doc:
            for page in doc:
                full_text += page.get_text()
        return full_text

    def _extract_patient_info(self, text: str) -> Dict[str, str]:
        """Extract patient demographics."""
        patient_info = {}
        
        # Extract name
        name_match = re.search(r'(?:Mr\.|Mrs\.|Ms\.|Dr\.)\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)', text)
        if name_match:
            patient_info['name'] = name_match.group(1).strip()
        
        # Extract age and sex
        age_sex_match = re.search(r'Age\s*/\s*Sex\s*:?\s*(\d+)\s*YRS?\s*/\s*([MF])', text, re.IGNORECASE)
        if age_sex_match:
            patient_info['age'] = age_sex_match.group(1)
            patient_info['sex'] = age_sex_match.group(2)
        
        # Extract registration number
        reg_match = re.search(r'Reg(?:\.|istration)?\s*no\.?\s*:?\s*(\d+)', text, re.IGNORECASE)
        if reg_match:
            patient_info['reg_no'] = reg_match.group(1)
        
        # Extract referred by
        ref_match = re.search(r'Referred\s*by\s*:?\s*([A-Za-z\s]+?)(?=\n|Reg)', text, re.IGNORECASE)
        if ref_match:
            patient_info['referred_by'] = ref_match.group(1).strip()
        
        return patient_info

    def _extract_dates(self, text: str) -> Dict[str, str]:
        """Extract all relevant dates."""
        dates = {}
        
        date_patterns = {
            'registered': r'Registered\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4}(?:\s+\d{1,2}:\d{2}\s*[AP]M)?)',
            'collected': r'Collected\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4})',
            'received': r'Received\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4})',
            'reported': r'Reported\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4}(?:\s+\d{1,2}:\d{2}\s*[AP]M)?)'
        }
        
        for key, pattern in date_patterns.items():
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                dates[key] = match.group(1).strip()
        
        return dates

    def _extract_medical_staff(self, text: str) -> Dict[str, Dict[str, str]]:
        """Extract doctor and lab staff information."""
        staff = {}
        
        # Lab incharge
        lab_match = re.search(r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\s*\n\s*([A-Z]+),\s*Lab\s*Incharge', text)
        if lab_match:
            staff['lab_incharge'] = {
                'name': lab_match.group(1).strip(),
                'qualification': lab_match.group(2).strip()
            }
        
        # Pathologist
        path_match = re.search(r'Dr\.\s*([A-Z][a-z]+(?:\s+[A-Z]\.\s*[A-Z][a-z]+)+)\s*\n\s*([A-Z\s,]+Pathologist)', text)
        if path_match:
            staff['pathologist'] = {
                'name': path_match.group(1).strip(),
                'qualification': path_match.group(2).strip()
            }
        
        return staff

    def _determine_status(self, value: float, ref_range: str) -> str:
        """Determine if value is normal, high, or low based on reference range."""
        try:
            # Parse reference range
            if '-' in ref_range:
                parts = ref_range.split('-')
                low = float(parts[0].strip().replace(',', ''))
                high = float(parts[1].strip().replace(',', ''))
                
                if value < low:
                    return "low"
                elif value > high:
                    return "high"
                else:
                    return "normal"
            elif '<' in ref_range:
                max_val = float(ref_range.replace('<', '').strip())
                return "normal" if value < max_val else "high"
            elif '>' in ref_range:
                min_val = float(ref_range.replace('>', '').strip())
                return "normal" if value > min_val else "low"
        except:
            pass
        
        return "unknown"

    def _extract_test_results_with_ai(self, text: str) -> Dict[str, Any]:
        """Use Groq AI to extract test results."""
        if not self.groq_client:
            return self._extract_test_results_rule_based(text)
        
        try:
            prompt = f"""Extract ONLY the blood test results from this medical report text.

Return ONLY a JSON object with this EXACT structure:
{{
  "TEST_NAME": {{
    "value": "numeric_value",
    "unit": "unit",
    "reference_range": "range"
  }}
}}

Rules:
1. Preserve all decimal points exactly as shown
2. Only include actual test parameters (not headers or labels)
3. Use the exact test names from the report
4. For differential counts, nest them under "DIFFERENTIAL LEUCOCYTE COUNT"
5. Extract reference ranges exactly as shown

Text:
{text}

Return ONLY the JSON, no explanation."""

            response = self.groq_client.chat.completions.create(
                model=self.groq_model,
                messages=[
                    {"role": "system", "content": "You extract test results into clean JSON. Return only valid JSON with no additional text."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=2000
            )
            
            content = response.choices[0].message.content.strip()
            
            # Extract JSON from response
            json_match = re.search(r'\{.*\}', content, re.DOTALL)
            if json_match:
                results = json.loads(json_match.group(0))
                
                # Add status to each test
                for test_name, test_data in results.items():
                    if isinstance(test_data, dict) and 'value' in test_data:
                        try:
                            value = float(test_data['value'].replace(',', ''))
                            ref_range = test_data.get('reference_range', '')
                            test_data['status'] = self._determine_status(value, ref_range)
                        except:
                            test_data['status'] = 'unknown'
                    elif isinstance(test_data, dict):
                        # Handle nested tests (like differential count)
                        for sub_test_name, sub_test_data in test_data.items():
                            if isinstance(sub_test_data, dict) and 'value' in sub_test_data:
                                try:
                                    value = float(sub_test_data['value'].replace(',', ''))
                                    ref_range = sub_test_data.get('reference_range', '')
                                    sub_test_data['status'] = self._determine_status(value, ref_range)
                                except:
                                    sub_test_data['status'] = 'unknown'
                
                return results
        except Exception as e:
            self.logger.warning(f"AI extraction failed: {e}. Falling back to rule-based.")
        
        return self._extract_test_results_rule_based(text)

    def _extract_test_results_rule_based(self, text: str) -> Dict[str, Any]:
        """Fallback rule-based extraction of test results."""
        results = {}
        
        # Pattern for test lines: TEST_NAME value unit reference_range
        pattern = r'^([A-Z][A-Za-z\s\(\),]+?)\s+([LH])?\s*(\d+(?:,\d+)?(?:\.\d+)?)\s+([a-zA-Z/%]+(?:/[a-zA-Z]+)?)\s+([\d\.\s\-<>]+)$'
        
        lines = text.split('\n')
        current_section = None
        
        for line in lines:
            line = line.strip()
            
            # Detect section headers
            if 'DIFFERENTIAL' in line.upper():
                current_section = 'DIFFERENTIAL LEUCOCYTE COUNT'
                results[current_section] = {}
                continue
            
            match = re.match(pattern, line)
            if match:
                test_name = match.group(1).strip()
                abnormal = match.group(2)  # L or H
                value = match.group(3)
                unit = match.group(4)
                ref_range = match.group(5).strip()
                
                test_data = {
                    'value': value,
                    'unit': unit,
                    'reference_range': ref_range
                }
                
                # Add status
                try:
                    numeric_value = float(value.replace(',', ''))
                    test_data['status'] = self._determine_status(numeric_value, ref_range)
                except:
                    if abnormal == 'L':
                        test_data['status'] = 'low'
                    elif abnormal == 'H':
                        test_data['status'] = 'high'
                    else:
                        test_data['status'] = 'normal'
                
                if current_section:
                    results[current_section][test_name] = test_data
                else:
                    results[test_name] = test_data
        
        return results

    def _extract_report_type(self, text: str) -> str:
        """Determine report type."""
        text_upper = text.upper()
        
        if 'COMPLETE BLOOD COUNT' in text_upper or 'CBC' in text_upper:
            return "Complete Blood Count (CBC)"
        elif 'LIPID' in text_upper:
            return "Lipid Profile"
        elif 'LIVER' in text_upper or 'LFT' in text_upper:
            return "Liver Function Test"
        elif 'KIDNEY' in text_upper or 'KFT' in text_upper:
            return "Kidney Function Test"
        elif 'THYROID' in text_upper:
            return "Thyroid Function Test"
        else:
            return "Medical Report"

    def _extract_clinical_notes(self, text: str) -> str:
        """Extract clinical notes section."""
        notes_match = re.search(r'Clinical\s+Notes?:\s*\n(.*?)(?=\n\n|Possible causes|$)', text, re.IGNORECASE | re.DOTALL)
        if notes_match:
            return notes_match.group(1).strip()
        return ""

    def extract_structured_data(self) -> Dict[str, Any]:
        """Extract clean, structured data from medical PDF."""
        self.logger.info(f"Processing {self.pdf_path}")
        
        # Extract full text
        full_text = self._extract_full_text()
        
        # Extract report date (first occurrence is usually the main report date)
        report_date = None
        date_match = re.search(r'\d{1,2}/\d{1,2}/\d{4}', full_text)
        if date_match:
            report_date = date_match.group(0)
        
        # Build structured output
        structured_data = {
            'document_info': {
                'filename': self.pdf_path.name,
                'report_type': self._extract_report_type(full_text),
                'report_date': report_date,
                'extraction_timestamp': datetime.now().isoformat()
            },
            'patient': self._extract_patient_info(full_text),
            'report_dates': self._extract_dates(full_text),
            'medical_staff': self._extract_medical_staff(full_text),
            'test_results': self._extract_test_results_with_ai(full_text),
            'clinical_notes': self._extract_clinical_notes(full_text)
        }
        
        self.logger.info("Extraction completed")
        return structured_data

    def save_to_json(self, output_path: str) -> None:
        """Extract and save to JSON."""
        data = self.extract_structured_data()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        self.logger.info(f"Data saved to {output_path}")


def main(pdf_path: str = "cbc-report-format.pdf", 
         output_path: str = "clean_extracted_data.json"):
    """Main function for easy usage."""
    
    extractor = StreamlinedMedicalOCR(pdf_path)
    extractor.save_to_json(output_path)
    
    print(f"✓ Extraction complete! Output saved to: {output_path}")


if __name__ == "__main__":
    main()

2025-12-20 04:11:06,788 - INFO - Processing cbc-report-format.pdf
2025-12-20 04:11:09,176 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:11:09,179 - INFO - Extraction completed
2025-12-20 04:11:09,185 - INFO - Data saved to clean_extracted_data.json


✓ Extraction complete! Output saved to: clean_extracted_data.json


In [ ]:
import json
import logging
import os
import re
from pathlib import Path
from typing import Dict, Any, Optional
from datetime import datetime

import fitz  # PyMuPDF
from groq import Groq
from dotenv import load_dotenv


class StreamlinedMedicalOCR:
    """
    Streamlined medical OCR that extracts only essential information.
    Removes redundant data and focuses on actionable medical information.
    """

    def __init__(self, pdf_path: str, groq_model: str = "meta-llama/llama-4-scout-17b-16e-instruct"):
        self.pdf_path = Path(pdf_path)
        self.groq_model = groq_model
        self._setup_logging()
        self._setup_groq_client()

    def _setup_logging(self) -> None:
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)

    def _setup_groq_client(self) -> None:
        load_dotenv()
        groq_api_key = os.getenv("GROQ_API_KEY")
        
        if not groq_api_key:
            self.logger.warning("GROQ_API_KEY not found. Using rule-based extraction only.")
            self.groq_client = None
        else:
            self.groq_client = Groq(api_key=groq_api_key)

    def _extract_full_text(self) -> str:
        """Extract all text from PDF."""
        full_text = ""
        with fitz.open(self.pdf_path) as doc:
            for page in doc:
                full_text += page.get_text()
        return full_text

    def _extract_patient_info(self, text: str) -> Dict[str, str]:
        """Extract patient demographics."""
        patient_info = {}
        
        # Extract name - look for Mr./Mrs./Ms./Dr. followed by name, but stop at newline or "Age"
        name_match = re.search(r'(?:Mr\.|Mrs\.|Ms\.|Dr\.)\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+?)(?=\s*\n|\s*Age)', text)
        if name_match:
            patient_info['name'] = name_match.group(1).strip()
        
        # Extract age and sex
        age_sex_match = re.search(r'Age\s*/\s*Sex\s*:?\s*(\d+)\s*YRS?\s*/\s*([MF])', text, re.IGNORECASE)
        if age_sex_match:
            patient_info['age'] = age_sex_match.group(1)
            patient_info['sex'] = age_sex_match.group(2)
        
        # Extract registration number
        reg_match = re.search(r'Reg(?:\.|istration)?\s*no\.?\s*:?\s*(\d+)', text, re.IGNORECASE)
        if reg_match:
            patient_info['reg_no'] = reg_match.group(1)
        
        # Extract referred by
        ref_match = re.search(r'Referred\s*by\s*:?\s*([A-Za-z\s]+?)(?=\n|Reg)', text, re.IGNORECASE)
        if ref_match:
            patient_info['referred_by'] = ref_match.group(1).strip()
        
        return patient_info

    def _extract_dates(self, text: str) -> Dict[str, str]:
        """Extract all relevant dates."""
        dates = {}
        
        date_patterns = {
            'registered': r'Registered\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4}(?:\s+\d{1,2}:\d{2}\s*[AP]M)?)',
            'collected': r'Collected\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4})',
            'received': r'Received\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4})',
            'reported': r'Reported\s+on\s*:?\s*(\d{1,2}/\d{1,2}/\d{4}(?:\s+\d{1,2}:\d{2}\s*[AP]M)?)'
        }
        
        for key, pattern in date_patterns.items():
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                dates[key] = match.group(1).strip()
        
        return dates

    def _extract_medical_staff(self, text: str) -> Dict[str, Dict[str, str]]:
        """Extract doctor and lab staff information."""
        staff = {}
        
        # Lab incharge - multiple patterns
        lab_patterns = [
            r'Mr\.\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\s*\n\s*([A-Z]+),\s*Lab\s*Incharge',
            r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\s*\n\s*([A-Z]+),\s*Lab\s*Incharge'
        ]
        for pattern in lab_patterns:
            lab_match = re.search(pattern, text)
            if lab_match:
                staff['lab_incharge'] = {
                    'name': lab_match.group(1).strip(),
                    'qualification': lab_match.group(2).strip()
                }
                break
        
        # Pathologist - multiple patterns
        path_patterns = [
            r'Dr\.\s*([A-Z]\.\s*[A-Z]\.\s*[A-Z][a-z]+)\s*\n\s*([A-Z\s,]+Pathologist)',
            r'Dr\.\s*([A-Z][a-z]+(?:\s+[A-Z]\.\s*[A-Z][a-z]+)+)\s*\n\s*([A-Z\s,]+Pathologist)',
            r'Dr\.\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\s*\n\s*([A-Z\s,]+Pathologist)'
        ]
        for pattern in path_patterns:
            path_match = re.search(pattern, text)
            if path_match:
                staff['pathologist'] = {
                    'name': path_match.group(1).strip(),
                    'qualification': path_match.group(2).strip()
                }
                break
        
        return staff

    def _determine_status(self, value: float, ref_range: str) -> str:
        """Determine if value is normal, high, or low based on reference range."""
        try:
            # Remove commas from reference range
            ref_range = ref_range.replace(',', '')
            
            # Parse reference range
            if '-' in ref_range:
                parts = ref_range.split('-')
                low = float(parts[0].strip())
                high = float(parts[1].strip())
                
                if value < low:
                    return "low"
                elif value > high:
                    return "high"
                else:
                    return "normal"
            elif '<' in ref_range:
                max_val = float(ref_range.replace('<', '').strip())
                return "normal" if value < max_val else "high"
            elif '>' in ref_range:
                min_val = float(ref_range.replace('>', '').strip())
                return "normal" if value > min_val else "low"
        except:
            pass
        
        return "unknown"

    def _extract_test_results_with_ai(self, text: str) -> Dict[str, Any]:
        """Use Groq AI to extract test results."""
        if not self.groq_client:
            return self._extract_test_results_rule_based(text)
        
        try:
            prompt = f"""Extract ONLY the blood test results from this medical report text.

Return ONLY a JSON object with this EXACT structure:
{{
  "TEST_NAME": {{
    "value": "numeric_value",
    "unit": "unit",
    "reference_range": "range"
  }}
}}

Rules:
1. Preserve all decimal points exactly as shown
2. Only include actual test parameters (not headers or labels)
3. Use the exact test names from the report
4. For differential counts, nest them under "DIFFERENTIAL LEUCOCYTE COUNT"
5. Extract reference ranges exactly as shown

Text:
{text}

Return ONLY the JSON, no explanation."""

            response = self.groq_client.chat.completions.create(
                model=self.groq_model,
                messages=[
                    {"role": "system", "content": "You extract test results into clean JSON. Return only valid JSON with no additional text."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=2000
            )
            
            content = response.choices[0].message.content.strip()
            
            # Extract JSON from response
            json_match = re.search(r'\{.*\}', content, re.DOTALL)
            if json_match:
                results = json.loads(json_match.group(0))
                
                # Add status to each test
                for test_name, test_data in results.items():
                    if isinstance(test_data, dict) and 'value' in test_data:
                        try:
                            value = float(test_data['value'].replace(',', ''))
                            ref_range = test_data.get('reference_range', '')
                            test_data['status'] = self._determine_status(value, ref_range)
                        except:
                            test_data['status'] = 'unknown'
                    elif isinstance(test_data, dict):
                        # Handle nested tests (like differential count)
                        for sub_test_name, sub_test_data in test_data.items():
                            if isinstance(sub_test_data, dict) and 'value' in sub_test_data:
                                try:
                                    value = float(sub_test_data['value'].replace(',', ''))
                                    ref_range = sub_test_data.get('reference_range', '')
                                    sub_test_data['status'] = self._determine_status(value, ref_range)
                                except:
                                    sub_test_data['status'] = 'unknown'
                
                return results
        except Exception as e:
            self.logger.warning(f"AI extraction failed: {e}. Falling back to rule-based.")
        
        return self._extract_test_results_rule_based(text)

    def _extract_test_results_rule_based(self, text: str) -> Dict[str, Any]:
        """Fallback rule-based extraction of test results."""
        results = {}
        
        # Pattern for test lines: TEST_NAME value unit reference_range
        pattern = r'^([A-Z][A-Za-z\s\(\),]+?)\s+([LH])?\s*(\d+(?:,\d+)?(?:\.\d+)?)\s+([a-zA-Z/%]+(?:/[a-zA-Z]+)?)\s+([\d\.\s\-<>]+)$'
        
        lines = text.split('\n')
        current_section = None
        
        for line in lines:
            line = line.strip()
            
            # Detect section headers
            if 'DIFFERENTIAL' in line.upper():
                current_section = 'DIFFERENTIAL LEUCOCYTE COUNT'
                results[current_section] = {}
                continue
            
            match = re.match(pattern, line)
            if match:
                test_name = match.group(1).strip()
                abnormal = match.group(2)  # L or H
                value = match.group(3)
                unit = match.group(4)
                ref_range = match.group(5).strip()
                
                test_data = {
                    'value': value,
                    'unit': unit,
                    'reference_range': ref_range
                }
                
                # Add status
                try:
                    numeric_value = float(value.replace(',', ''))
                    test_data['status'] = self._determine_status(numeric_value, ref_range)
                except:
                    if abnormal == 'L':
                        test_data['status'] = 'low'
                    elif abnormal == 'H':
                        test_data['status'] = 'high'
                    else:
                        test_data['status'] = 'normal'
                
                if current_section:
                    results[current_section][test_name] = test_data
                else:
                    results[test_name] = test_data
        
        return results

    def _extract_report_type(self, text: str) -> str:
        """Determine report type."""
        text_upper = text.upper()
        
        if 'COMPLETE BLOOD COUNT' in text_upper or 'CBC' in text_upper:
            return "Complete Blood Count (CBC)"
        elif 'LIPID' in text_upper:
            return "Lipid Profile"
        elif 'LIVER' in text_upper or 'LFT' in text_upper:
            return "Liver Function Test"
        elif 'KIDNEY' in text_upper or 'KFT' in text_upper:
            return "Kidney Function Test"
        elif 'THYROID' in text_upper:
            return "Thyroid Function Test"
        else:
            return "Medical Report"

    def _extract_clinical_notes(self, text: str) -> str:
        """Extract clinical notes section."""
        notes_match = re.search(r'Clinical\s+Notes?:\s*\n(.*?)(?=\n\n|Possible causes|$)', text, re.IGNORECASE | re.DOTALL)
        if notes_match:
            return notes_match.group(1).strip()
        return ""

    def extract_structured_data(self) -> Dict[str, Any]:
        """Extract clean, structured data from medical PDF."""
        self.logger.info(f"Processing {self.pdf_path}")
        
        # Extract full text
        full_text = self._extract_full_text()
        
        # Extract report date (first occurrence is usually the main report date)
        report_date = None
        date_match = re.search(r'\d{1,2}/\d{1,2}/\d{4}', full_text)
        if date_match:
            report_date = date_match.group(0)
        
        # Build structured output
        structured_data = {
            'document_info': {
                'filename': self.pdf_path.name,
                'report_type': self._extract_report_type(full_text),
                'report_date': report_date,
                'extraction_timestamp': datetime.now().isoformat()
            },
            'patient': self._extract_patient_info(full_text),
            'report_dates': self._extract_dates(full_text),
            'medical_staff': self._extract_medical_staff(full_text),
            'test_results': self._extract_test_results_with_ai(full_text),
            'clinical_notes': self._extract_clinical_notes(full_text)
        }
        
        self.logger.info("Extraction completed")
        return structured_data

    def save_to_json(self, output_path: str) -> None:
        """Extract and save to JSON."""
        data = self.extract_structured_data()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        self.logger.info(f"Data saved to {output_path}")


def main(pdf_path: str = r"C:\Users\laksh\Downloads\PDF MAIN!@\Medical_final_project_github_folder\cbc-report-format.pdf", 
         output_path: str = "clean_extracted_data.json"):
    """Main function for easy usage."""
    
    extractor = StreamlinedMedicalOCR(pdf_path)
    extractor.save_to_json(output_path)
    
    print(f"✓ Extraction complete! Output saved to: {output_path}")


if __name__ == "__main__":
    main()

2025-12-20 04:10:16,432 - INFO - Processing C:\Users\laksh\Downloads\PDF MAIN!@\Medical_final_project_github_folder\cbc-report-format.pdf
2025-12-20 04:10:17,148 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-20 04:10:17,151 - INFO - Extraction completed
2025-12-20 04:10:17,153 - INFO - Data saved to clean_extracted_data.json


✓ Extraction complete! Output saved to: clean_extracted_data.json


In [28]:
pip install pdfplumber==0.11.0

  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 5.6/5.6 MB 34.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 35.0 MB/s eta 0:00:00
Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl (183 kB)
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 45.0 MB/s eta 0:00:00
Using cached pycparser-2.23-py3-none-any.whl (118 kB)

   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   -------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import logging
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
from contextlib import contextmanager

import cv2
import numpy as np
import pytesseract
import pdfplumber  # CRITICAL NEW LIBRARY
from pdf2image import convert_from_path
import fitz  # PyMuPDF
from groq import Groq
from dotenv import load_dotenv

# --- DATA STRUCTURES ---
@dataclass
class TextBlock:
    text: str
    x0: float
    top: float
    x1: float
    bottom: float

@dataclass
class ExtractedTable:
    extracted_source: str  # "pdfplumber" or "ocr"
    raw_text: str
    structured_data: List[Dict[str, Any]]
    page_number: int

@dataclass
class MedicalReport:
    filename: str
    metadata: Dict[str, Any]
    results: List[Dict[str, Any]]
    clinical_notes: str

class RobustMedicalOCR:
    def __init__(self, pdf_path: str, groq_api_key: Optional[str] = None):
        self.pdf_path = Path(pdf_path)
        self.groq_client = Groq(api_key=groq_api_key) if groq_api_key else None
        self._setup_logging()

    def _setup_logging(self):
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)

    # --- STRATEGY 1: TRY DIGITAL EXTRACTION FIRST (Most Accurate) ---
    def _extract_tables_digital(self, page) -> List[ExtractedTable]:
        """
        Uses pdfplumber to extract tables from digital PDFs.
        Handles tables with and without visible gridlines.
        """
        tables = []
        
        # Strategy A: Explicit Table Extraction (for bordered tables)
        extracted_tables = page.extract_tables()
        
        # Strategy B: Visual Debugging / Whitespace analysis settings
        table_settings = {
            "vertical_strategy": "text", 
            "horizontal_strategy": "text",
            "intersection_x_tolerance": 15
        }
        
        if not extracted_tables:
            # Fallback for whitespace tables
            extracted_tables = page.extract_tables(table_settings)

        if extracted_tables:
            for table_data in extracted_tables:
                # Clean None values
                cleaned_data = [[cell if cell else "" for cell in row] for row in table_data]
                
                # Convert list of lists to structured dict immediately
                structured = self._structure_tabular_data(cleaned_data)
                
                # Serialize for LLM context
                raw_text = "\n".join([" | ".join(map(str, row)) for row in cleaned_data])
                
                tables.append(ExtractedTable(
                    extracted_source="pdfplumber_digital",
                    raw_text=raw_text,
                    structured_data=structured,
                    page_number=page.page_number
                ))
        
        return tables

    # --- STRATEGY 2: ROW CLUSTERING OCR (For Scanned Images) ---
    def _cluster_text_into_rows(self, data: Dict) -> List[List[str]]:
        """
        CRITICAL: Groups OCR words into logical rows based on Y-coordinates.
        Solves the issue where values are fragmented from their labels.
        """
        n_boxes = len(data['text'])
        blocks = []
        
        # Filter out empty text and low confidence
        for i in range(n_boxes):
            if int(data['conf'][i]) > 30 and data['text'][i].strip():
                blocks.append({
                    "text": data['text'][i],
                    "x": data['left'][i],
                    "y": data['top'][i],
                    "h": data['height'][i]
                })

        # Sort by Y coordinate
        blocks.sort(key=lambda b: b['y'])

        rows = []
        if not blocks:
            return rows

        current_row = [blocks[0]]
        # Dynamic row height threshold (half the height of current text)
        y_threshold = blocks[0]['h'] / 2 

        for block in blocks[1:]:
            last_block = current_row[-1]
            # Check if this block is on the same line as the last one
            if abs(block['y'] - last_block['y']) < y_threshold:
                current_row.append(block)
            else:
                # Sort the completed row by X coordinate (read left to right)
                current_row.sort(key=lambda b: b['x'])
                rows.append(current_row)
                current_row = [block]
        
        # Append last row
        if current_row:
            current_row.sort(key=lambda b: b['x'])
            rows.append(current_row)

        # Convert blocks to simple strings
        text_rows = []
        for row in rows:
            # Combine words, handling large gaps as column separators
            row_str = []
            for i, block in enumerate(row):
                if i > 0 and (block['x'] - (row[i-1]['x'] + len(row[i-1]['text']) * 10)) > 20:
                     row_str.append("|") # Visual separator
                row_str.append(block['text'])
            text_rows.append(" ".join(row_str))
            
        return text_rows

    def _extract_tables_ocr_fallback(self, image_path: str, page_num: int) -> List[ExtractedTable]:
        """
        Fallback for Scanned Docs: Preprocessing -> Tesseract Data -> Row Clustering
        """
        self.logger.info(f"Using OCR fallback for page {page_num}")
        image = cv2.imread(image_path)
        
        # 1. Preprocessing (Grayscale -> Binary -> De-skew)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
        
        # 2. Get Layout Data (not just string)
        custom_config = r'--oem 3 --psm 6'
        data = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
        
        # 3. Cluster into Rows
        text_rows = self._cluster_text_into_rows(data)
        
        raw_text = "\n".join(text_rows)
        
        return [ExtractedTable(
            extracted_source="ocr_fallback",
            raw_text=raw_text,
            structured_data=[], # Will be filled by LLM later
            page_number=page_num
        )]

    # --- PROCESSING PIPELINE ---
    def process_document(self) -> Dict[str, Any]:
        full_text_context = ""
        all_tables = []
        
        # Open with PDFPlumber
        with pdfplumber.open(self.pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                # Try Digital Extraction
                tables = self._extract_tables_digital(page)
                
                # Check if page is scanned (empty digital text)
                text_density = len(page.extract_text() or "")
                
                if not tables and text_density < 50:
                    # RENDER IMAGE FOR OCR
                    # Note: In production, convert specific page bytes to image
                    # Here we use pdf2image for simplicity
                    images = convert_from_path(str(self.pdf_path), first_page=i+1, last_page=i+1)
                    temp_img = f"temp_page_{i}.jpg"
                    images[0].save(temp_img)
                    
                    tables = self._extract_tables_ocr_fallback(temp_img, i+1)
                    os.remove(temp_img)
                
                all_tables.extend(tables)
                
                # Aggregate text for Context
                for t in tables:
                    full_text_context += f"\n--- Page {i+1} Table ---\n{t.raw_text}"

        # Final AI Parsing
        return self._ai_parse_results(full_text_context)

    def _structure_tabular_data(self, rows: List[List[str]]) -> List[Dict]:
        """Convert list of lists into potential key-value pairs heuristically"""
        structured = []
        headers = []
        
        # Heuristic: Find header row
        for row in rows:
            row_str = " ".join(row).lower()
            if "test" in row_str and ("result" in row_str or "value" in row_str):
                headers = [h.lower() for h in row]
                continue
            
            # Simple mapping if no headers found yet
            if not headers:
                if len(row) >= 2:
                    structured.append({"parameter": row[0], "value": row[1]})
            else:
                # Map headers to values
                item = {}
                for idx, cell in enumerate(row):
                    if idx < len(headers):
                        item[headers[idx]] = cell
                structured.append(item)
                
        return structured

    def _ai_parse_results(self, context_text: str) -> Dict[str, Any]:
        """
        Sends the clustered, cleaned text to LLM for final structuring.
        This is robust because the input context is now row-aligned.
        """
        if not self.groq_client:
            return {"error": "No LLM Client", "raw_context": context_text}
            
        prompt = f"""
        Analyze the following text extracted from a Medical Lab Report.
        The text has been pre-processed to grouping distinct rows.
        
        EXTRACT THE FOLLOWING JSON STRUCTURE:
        {{
            "patient": {{ "name": "", "age": "", "gender": "", "id": "" }},
            "results": [
                {{
                    "test_name": "exact parameter name",
                    "value": "numeric value only",
                    "unit": "g/dL, %, etc",
                    "flag": "High/Low/Normal",
                    "reference_range": "extracted range"
                }}
            ],
            "notes": "summary of clinical notes"
        }}

        RULES:
        1. If a value is split (e.g. "5" "100"), infer the full number based on context (5100).
        2. Look for flags like 'H', 'L', 'High', 'Low' in adjacent columns.
        3. Do not invent data.

        INPUT TEXT:
        {context_text}
        """

        try:
            response = self.groq_client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct", # Excellent for reasoning
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            self.logger.error(f"AI Parsing failed: {e}")
            return {"raw_text": context_text}

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    load_dotenv()
    api_key = os.getenv("GROQ_API_KEY")
    
    # Run the extractor
    extractor = RobustMedicalOCR("CBC-test-report-format-example-sample-template-Drlogy-lab-report.pdf", groq_api_key=api_key)
    data = extractor.process_document()
    
    # --- ADD THIS PART TO SAVE TO FILE ---
    output_filename = "medical_ocr_output.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Success! Data saved to: {output_filename}")

2025-12-20 04:27:03,183 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Success! Data saved to: medical_ocr_output.json


In [ ]:
import json
import logging
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
from contextlib import contextmanager

import cv2
import numpy as np
import pytesseract
import pdfplumber
from pdf2image import convert_from_path
import fitz  # PyMuPDF
from groq import Groq
from dotenv import load_dotenv

# --- DATA STRUCTURES ---
@dataclass
class TextBlock:
    text: str
    x0: float
    top: float
    x1: float
    bottom: float

@dataclass
class ExtractedTable:
    extracted_source: str
    raw_text: str
    structured_data: List[Dict[str, Any]]
    page_number: int

@dataclass
class MedicalReport:
    filename: str
    metadata: Dict[str, Any]
    results: List[Dict[str, Any]]
    clinical_notes: str

class RobustMedicalOCR:
    def __init__(self, pdf_path: str, groq_api_key: Optional[str] = None):
        self.pdf_path = Path(pdf_path)
        self.groq_client = Groq(api_key=groq_api_key) if groq_api_key else None
        self._setup_logging()

    def _setup_logging(self):
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)

    # --- STRATEGY 1: DIGITAL EXTRACTION ---
    def _extract_tables_digital(self, page) -> List[ExtractedTable]:
        tables = []
        extracted_tables = page.extract_tables()
        
        table_settings = {
            "vertical_strategy": "text", 
            "horizontal_strategy": "text",
            "intersection_x_tolerance": 15
        }
        
        if not extracted_tables:
            extracted_tables = page.extract_tables(table_settings)

        if extracted_tables:
            for table_data in extracted_tables:
                cleaned_data = [[cell if cell else "" for cell in row] for row in table_data]
                structured = self._structure_tabular_data(cleaned_data)
                raw_text = "\n".join([" | ".join(map(str, row)) for row in cleaned_data])
                
                tables.append(ExtractedTable(
                    extracted_source="pdfplumber_digital",
                    raw_text=raw_text,
                    structured_data=structured,
                    page_number=page.page_number
                ))
        return tables

    # --- STRATEGY 2: OCR FALLBACK ---
    def _cluster_text_into_rows(self, data: Dict) -> List[List[str]]:
        n_boxes = len(data['text'])
        blocks = []
        for i in range(n_boxes):
            if int(data['conf'][i]) > 30 and data['text'][i].strip():
                blocks.append({
                    "text": data['text'][i],
                    "x": data['left'][i],
                    "y": data['top'][i],
                    "h": data['height'][i]
                })

        blocks.sort(key=lambda b: b['y'])
        rows = []
        if not blocks:
            return rows

        current_row = [blocks[0]]
        y_threshold = blocks[0]['h'] / 2 

        for block in blocks[1:]:
            last_block = current_row[-1]
            if abs(block['y'] - last_block['y']) < y_threshold:
                current_row.append(block)
            else:
                current_row.sort(key=lambda b: b['x'])
                rows.append(current_row)
                current_row = [block]
        
        if current_row:
            current_row.sort(key=lambda b: b['x'])
            rows.append(current_row)

        text_rows = []
        for row in rows:
            row_str = []
            for i, block in enumerate(row):
                if i > 0 and (block['x'] - (row[i-1]['x'] + len(row[i-1]['text']) * 10)) > 20:
                     row_str.append("|")
                row_str.append(block['text'])
            text_rows.append(" ".join(row_str))
            
        return text_rows

    def _extract_tables_ocr_fallback(self, image_path: str, page_num: int) -> List[ExtractedTable]:
        self.logger.info(f"Using OCR fallback for page {page_num}")
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
        
        custom_config = r'--oem 3 --psm 6'
        data = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
        text_rows = self._cluster_text_into_rows(data)
        raw_text = "\n".join(text_rows)
        
        return [ExtractedTable(
            extracted_source="ocr_fallback",
            raw_text=raw_text,
            structured_data=[],
            page_number=page_num
        )]

    # --- PROCESSING PIPELINE ---
    def process_document(self) -> Dict[str, Any]:
        full_text_context = ""
        
        with pdfplumber.open(self.pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                # 1. Capture Header/Footer/Misc Text (Digital) 
                # ### NEW: This captures doctor names outside of tables
                raw_page_text = page.extract_text() or ""
                full_text_context += f"\n--- Page {i+1} Raw Text (Header/Footer) ---\n{raw_page_text}\n"

                # 2. Extract Tables (Digital)
                tables = self._extract_tables_digital(page)
                
                # 3. Check for Scanned Page
                # If page is mostly empty or tables failed, assume scanned
                if not tables and len(raw_page_text) < 50:
                    images = convert_from_path(str(self.pdf_path), first_page=i+1, last_page=i+1)
                    temp_img = f"temp_page_{i}.jpg"
                    images[0].save(temp_img)
                    
                    # OCR extracts everything (header + tables together)
                    ocr_tables = self._extract_tables_ocr_fallback(temp_img, i+1)
                    for t in ocr_tables:
                        full_text_context += f"\n--- Page {i+1} OCR Text ---\n{t.raw_text}"
                    os.remove(temp_img)
                else:
                    # Append structured table text for clarity
                    for t in tables:
                        full_text_context += f"\n--- Page {i+1} Table Data ---\n{t.raw_text}"

        return self._ai_parse_results(full_text_context)

    def _structure_tabular_data(self, rows: List[List[str]]) -> List[Dict]:
        structured = []
        headers = []
        for row in rows:
            row_str = " ".join(row).lower()
            if "test" in row_str and ("result" in row_str or "value" in row_str):
                headers = [h.lower() for h in row]
                continue
            
            if not headers:
                if len(row) >= 2:
                    structured.append({"parameter": row[0], "value": row[1]})
            else:
                item = {}
                for idx, cell in enumerate(row):
                    if idx < len(headers):
                        item[headers[idx]] = cell
                structured.append(item)
        return structured

    def _ai_parse_results(self, context_text: str) -> Dict[str, Any]:
        if not self.groq_client:
            return {"error": "No LLM Client", "raw_context": context_text}
            
        # ### UPDATED PROMPT: Added 'doctor' and 'lab_details' to JSON structure
        prompt = f"""
        Analyze the following text extracted from a Medical Lab Report.
        
        EXTRACT THE FOLLOWING JSON STRUCTURE:
        {{
            "patient": {{ 
                "name": "Extract patient name", 
                "age": "Extract age", 
                "gender": "Extract gender", 
                "id": "Extract Patient ID/MRN" 
            }},
            "doctor": {{
                "name": "Name of referring doctor or pathologist",
                "signature_text": "Text found near signature (e.g. 'Dr. Amit', 'MD Pathology')"
            }},
            "lab_details": {{
                "name": "Name of the lab/hospital",
                "report_date": "Date of report generation"
            }},
            "results": [
                {{
                    "test_name": "exact parameter name",
                    "value": "numeric value only",
                    "unit": "g/dL, %, etc",
                    "flag": "High/Low/Normal/Critical",
                    "reference_range": "extracted range"
                }}
            ],
            "clinical_notes": "summary of remarks or interpretation"
        }}

        RULES:
        1. Doctor names are often at the bottom (signature area) or top (referral).
        2. If multiple doctors are listed, prioritize the one signing the report.
        3. Infer full numbers if spaces exist (e.g. "1 50" -> 150).
        4. Return ONLY valid JSON.

        INPUT TEXT:
        {context_text}
        """

        try:
            response = self.groq_client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct", # Switched to 70b for better detail extraction
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            self.logger.error(f"AI Parsing failed: {e}")
            return {"raw_text": context_text}

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    load_dotenv()
    api_key = os.getenv("GROQ_API_KEY")
    
    # Run the extractor
    # Make sure you have a valid PDF path here
    extractor = RobustMedicalOCR("LabReport.pdf", groq_api_key=api_key)
    data = extractor.process_document()
    
    output_filename = "medical_ocr_output.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Extraction Complete. Data saved to {output_filename}")

2026-01-28 01:11:03,616 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Extraction Complete. Data saved to medical_ocr_output.json
